# 4.0 本章表格的讀法

全章各表共用同一組欄位。先一次講清楚，後續不再重複。

| 欄位 | 意義 | 如何得到 |
| :--- | :--- | :--- |
| **pp**（百分點） | **兩個百分比相減**的單位。年化 1.5% 對 0.7% 的差是 **0.8 pp**，不是 0.8% | 差值本身 |
| **年化Δ** | 處理組減對照組的**年化報酬差**，正值代表處理組較優 | 逐日差分 $\Delta r_t$ 的平均 × 252 |
| **95% CI** | 效果量的信賴區間 | bootstrap 分布的 2.5／97.5 百分位（§3.5） |
| **$p$** | 雙尾 $p$ 值 | 將 bootstrap 分布平移至中心為零後，觀測值的極端程度 |
| **BH 校正 $p$** | 多重比較校正後的 $p$ | Benjamini-Hochberg 控制 FDR；同一命題的各組一起校正 |
| **IR**（資訊比率） | 差分序列的**風險調整後**效果 | 年化Δ ÷ 差分序列的年化標準差 |
| **勝日%** | 處理組當日報酬高於對照組的交易日占比 | $\#\{\Delta r_t > 0\}$ ÷ 交易日數 |
| **逐格顯著** | 15 個參數格中，**單獨**檢定即達 5% 顯著者的格數 | 每格各跑一次 bootstrap |

::: {.callout-important}

### 兩個最容易誤讀的欄位

**「pp」不是「%」。** 本研究的策略絕對年化報酬多落在 −0.4% 至 +0.8%，
而命題 2 的效果量是 +0.787 **pp**——**效果量比報酬水準本身還大**。
這不矛盾：Δ 是兩臂相減，共同的市場成分已被消去（§3.5）。

**「逐格顯著 6/15」不是「只有 6 格有效」。** 15 格全部方向為正卻只有 6 格
單獨顯著，代表的是**單格樣本不足以偵測**，而非效果只出現在那 6 格。
主張一律以等權組合為準（§3.5 報告口徑）。

:::

::: {.aside}
另有 $SE$（標準誤）與 MDE（最小可偵測效果）兩個量出現於 §4.1.3 與 §4.5，
其定義與推導於該處給出。
:::


# 4.1 命題 1：ML 分群 vs GICS

## 主檢定：9 組對照，校正後無一顯著

固定排序準則與交易端，唯一變因為分組方法。$\Delta = r_{ML} - r_{GICS}$，正值支持命題 1。

| 分群 | 排序 | 年化Δ(pp) | 95% CI (pp) | 方向 | $p$ | BH 校正 $p$ |
| :--- | :--- | ---: | :---: | :---: | ---: | ---: |
| Agglomerative | SSD | **+0.369** | [−0.44, +1.15] | ML 優 | 0.363 | 0.545 |
| HDBSCAN | SSD | **+0.368** | [−0.26, +1.02] | ML 優 | 0.261 | 0.545 |
| HDBSCAN | DTW | +0.246 | [−0.41, +0.93] | ML 優 | 0.468 | 0.602 |
| HDBSCAN | SDP | +0.160 | [−0.41, +0.77] | ML 優 | 0.598 | 0.673 |
| K-means | SSD | +0.025 | [−0.97, +0.94] | ML 優 | 0.954 | 0.954 |
| Agglomerative | DTW | −0.452 | [−1.32, +0.40] | GICS 優 | 0.305 | 0.545 |
| Agglomerative | SDP | −0.556 | [−1.32, +0.18] | GICS 優 | 0.143 | 0.455 |
| K-means | DTW | −0.647 | [−1.53, +0.23] | GICS 優 | 0.152 | 0.455 |
| K-means | SDP | −0.816 | [−1.71, +0.02] | GICS 優 | 0.063 | 0.455 |

**BH-FDR 校正後無一顯著**（校正後最小 $p$ = 0.455；校正前最小 0.063）。
方向 **ML 優 5 組、GICS 優 4 組**。

::: {.aside}
全程並行計算 Newey-West HAC 作為參數法對照，**14 組對照兩法結論完全一致**（命題 2 五組皆顯著、命題 1 九組皆不顯著）；完整雙欄見 `results/analysis/` 各 CSV。
:::


## 信賴區間：不顯著的性質是檢定力不足

::: {.callout-important}

雙尾檢定不顯著 **不等於** 兩者相當。區分兩者只需看**區間寬度**。

:::

**9 組區間全部涵蓋 0，寬度 1.2 ~ 1.9 pp**
（最寬 K-means × SSD：[−0.97, +0.94]，寬 1.91；
最窄 HDBSCAN × SDP：[−0.41, +0.77]，寬 1.18）。

而 GICS 參照臂**自身**的等權年化僅 −0.33% / −0.40% / −0.11%（SSD/DTW/SDP）
→ **區間寬度是參照臂整體績效的數倍**。

亦即資料同時無法排除「ML 優於 GICS 達 0.9 pp」與「劣於 GICS 達 1.0 pp」。

> **本研究對此比較檢定力不足**：既無法證明 ML 較優，亦無法主張兩者相當。
> 命題 1 **未獲支持**——此為「未能拒絕虛無假設」，**非**「GICS 顯著較優」。


## 檢定力的量化：兩層比較為何不對等

「檢定力不足」若只是定性陳述，讀者無從判斷嚴重程度。$p$ 值與信賴區間同源於同一次
重抽，故標準誤可由區間寬度直接還原（$SE = $ 區間寬度 $/\,3.92$），
據以計算 $\alpha$=5%、檢定力 80% 下的**最小可偵測效果**（MDE）：

$$\text{MDE} = (z_{\alpha/2} + z_{\beta}) \times SE = 2.80 \times SE$$

其中 $1.96\,SE$ 是宣告顯著所需的距離，額外的 $0.84\,SE$ 是為了讓真實效果有八成機率
落在該距離之外——真實效果若恰等於 $1.96\,SE$，只有**半數**機率被偵測到。

| | $SE$ 中位 (pp) | $\lvert\Delta\rvert$ 中位 (pp) | $\lvert t \rvert$ 中位 | MDE 中位 (pp) |
| :--- | ---: | ---: | ---: | ---: |
| 命題 1（形成期，兩臂持**不同**配對） | 0.406 | 0.369 | 1.03 | **1.14** |
| 命題 2（交易期，兩臂**共用**配對） | 0.214 | 0.787 | 3.32 | **0.60** |

::: {.callout-important}

### 三項可直接讀出的事實

1. **命題 2 的顯著不是僥倖。** 點估計中位 +0.787 pp 明顯大於自身 MDE 0.60 pp。
2. **命題 1 的不顯著幾乎不帶資訊。** 若真實優勢為 **0.3 pp**（已足以翻轉參照臂
   −0.11% ~ −0.40% 的損益方向），本檢定偵測到它的機率僅 **11%**
   ——即使命題 1 為真，本實驗仍有近九成機率報告「不顯著」。
3. **不對等來自設計，不是資料量。** 命題 2 兩臂共用配對，逐日差分消去共同市場衝擊；
   命題 1 兩臂持不同配對，差分混入兩套標的的特異變異。
   $SE$ 相差 **1.89 倍**，等效於形成期層需 **3.6 倍**樣本期間。

:::

> 這是結構性的：更換分群演算法不會改變它。完整意涵見 §4.5。


# 4.2 命題 2：門檻選擇 vs. 固定門檻

## 主檢定：五種配對底全部顯著

對照的兩臂為 **DL-THR**（逐期由模型選擇進場門檻）與 **Z-Score**（固定 $z$=2.0）。
兩臂共用同一批配對、同一組參數格，唯一變因是門檻如何決定。

| 配對底 | 年化Δ(pp) | 95% CI (pp) | 資訊比率 | 勝日% | $p$ | 逐格顯著 |
| :--- | ---: | :---: | ---: | ---: | ---: | :---: |
| **GICS × SSD（傳統）** | **+1.105** | [+0.58, +1.76] | 0.741 | 52.3 | **0.0010** | **8/15** |
| HDBSCAN × SDP | +0.865 | [+0.48, +1.26] | 0.575 | 52.6 | **0.0000** | 6/15 |
| Agglomerative × SSD | +0.787 | [+0.34, +1.27] | 0.583 | 51.7 | **0.0011** | 6/15 |
| GICS × SDP（傳統） | +0.659 | [+0.29, +1.07] | 0.507 | 51.7 | **0.0009** | 7/15 |
| K-means × SSD | +0.606 | [+0.20, +1.04] | 0.449 | 51.5 | **0.0060** | 5/15 |

::: {.callout-important}

**五個 95% CI 完全落在零的右側**——與命題 1 的九個區間全部橫跨零形成對比。

最保守的下界（K-means × SSD）仍為 **+0.20 pp**
→ 方向性主張在**最不利的區間端點**下依然成立。

:::

> **增益最大者為傳統 GICS 配對底**（+1.105 pp，且逐格顯著數最高 8/15）
> → 門檻選擇的改良與「配對如何找到」**正交**：它不依賴機器學習分群，
> 在傳統產業分組上反而取得最大增益。

::: {.aside}
全程並行計算 Newey-West HAC 作為參數法對照，**14 組對照兩法結論完全一致**（命題 2 五組皆顯著、命題 1 九組皆不顯著）；完整雙欄見 `results/analysis/` 各 CSV。
:::


## 單一參數配置的檢定力限制

等權組合顯著，係因平均掉各配置的特異噪音。逐格檢定：

| 配對底 | 15 格中正向顯著 | 年化Δ 中位 (pp) | 年化Δ 全距 (pp) |
| :--- | :---: | ---: | :---: |
| GICS-SSD | **8/15** | +0.830 | +0.14 ~ +3.44 |
| GICS-SDP | 7/15 | +0.588 | −1.18 ~ +3.29 |
| HDBSCAN | 6/15 | +0.646 | −0.49 ~ +4.48 |
| Agglomerative | 6/15 | +0.474 | −0.45 ~ +3.79 |
| K-means | 5/15 | +0.340 | −0.48 ~ +3.17 |

::: {.callout-important}

各配對底中位數皆為正、75 格中僅少數為負（GICS-SSD 底 15 格全正）→ **效果為真**；
但 **單一參數設定的檢定力不足** —— 15 格中僅三分之一至一半達顯著，
實務上僅執行一組設定時，有相當機率無法觀察到顯著改善。

此點必須據實揭露。

:::

> **全距欄同時說明本研究為何一律報告等權組合**：最佳格的年化增益高達
> +3.2 ~ +4.5 pp，遠高於等權組合的 +0.6 ~ +1.1——但最佳格**無法事前取得**（§3.5.5）。


## 增益來源①：不是拉高門檻

DL-THR 實際進場門檻中位數 **2.21–2.29**，高於基準 2.0。
若增益僅來自「少交易」，把 Z-Score 門檻拉到同一水準應能複製。

| 配對底 | A：DL-THR−ZS(2.0) | **B：DL-THR−ZS(2.2)** | C：門檻管道複製率 |
| :--- | ---: | ---: | ---: |
| Agglomerative | +0.787 (0.0011) | **+0.714 (0.0056)** | 9.3% ($p$=0.44) |
| HDBSCAN | +0.865 (0.0000) | **+0.619 (0.0052)** | **28.4% ($p$=0.014)** ⚠ |
| K-means | +0.606 (0.0060) | **+0.660 (0.0117)** | −8.9% ($p$=0.61) |
| GICS-SSD | +1.105 (0.0010) | **+1.060 (0.0024)** | 4.1% ($p$=0.65) |
| GICS-SDP | +0.659 (0.0009) | **+0.511 (0.0083)** | 22.5% ($p$=0.14) |

同門檻對照下**五種配對底仍全部顯著**——這是本項的結論，且不受下列揭露影響。

::: {.callout-warning}

### 揭露：門檻管道在 HDBSCAN 底下確實顯著

門檻管道複製的比例介於 **−8.9% 至 28.4%**，**五組中有一組達顯著**：
HDBSCAN 底的 ZS(2.2)−ZS(2.0) 為 **+0.246 pp（$p$=0.014）**，複製了 28.4% 的表面增益；
門檻拉到 2.5 時複製率達 **52.5%（$p$=0.023）**。

即便如此，DL-THR 在同門檻下仍顯著勝出（HDBSCAN 底 +0.619 pp，$p$=0.0052），
故「增益不僅是拉高門檻」的結論成立——**惟門檻管道並非毫無貢獻**。

:::

⚠️ 須揭露：對照門檻拉至 2.5（已超出 DL-THR 實際行為範圍）時，
五支中三支失去 5% 顯著性——HDBSCAN ($p$=0.145)、GICS-SDP ($p$=0.071)、
K-means ($p$=0.065)；Agglomerative ($p$=0.058) 落在邊緣，
僅 GICS-SSD ($p$=0.038) 維持顯著。


## 增益來源②：不是篩掉劣質配對

DL-THR 的 SKIP 率為 **32.0–33.1%**。

::: {.callout-important}

底層策略期望值為負 → **隨機跳過任一批配對，期望上都會「避開損失」**
→ 必須以置換檢定建立虛無分布。

:::

| 配對底 | 實際避損 | 隨機期望 | 技巧成分 | 顯著格數 |
| :--- | ---: | ---: | ---: | :---: |
| Agglomerative | 597.7 | 356.6 | +241.2 | 0/15 |
| HDBSCAN | 857.6 | 415.8 | +441.9 | 2/15 |
| K-means | 494.7 | 428.2 | +66.5 | 1/15 |
| GICS-SSD | 825.2 | 373.2 | +452.0 | 3/15 |
| GICS-SDP | 237.7 | 243.1 | **−5.4** | 0/15 |

**75 格中僅 6 格顯著**，而 $\alpha$=0.05 下純靠運氣的期望值為 **3.75 格**。
機械成分佔實際避損的 **45–102%**。

> 損益分解顯示「SKIP 貢獻 31–52%」——但該數字為**會計恆等式而非技巧歸因**，
> 不可據以宣稱模型學會篩選劣質配對。


## 增益來源③：更不是減少曝險

直覺上 SKIP 應使交易量下降。**資料顯示相反**（15 格平均）：

| 配對底 | DL-THR 進場次數 | Z-Score 進場次數 | 倍數 |
| :--- | ---: | ---: | ---: |
| Agglomerative | 7,715 | 4,488 | **1.72×** |
| HDBSCAN | 8,997 | 5,433 | **1.66×** |
| K-means | 4,853 | 3,125 | **1.55×** |
| GICS-SSD | 11,436 | 5,971 | **1.92×** |
| GICS-SDP | 11,162 | 6,005 | **1.86×** |

SKIP 掉一組配對會**釋放組合槽位給其他配對** → 淨效果為總交易次數**上升**。

同門檻對照亦確認曝險確實被拉齊：ZS(2.2) 的進場次數較 ZS(2.0) 僅降 4–7%，
而 DL-THR 仍是兩者的 1.55–1.92 倍——**增益不可能來自「少交易」**。

::: {.callout-tip}

### 三項行為面替代解釋皆被排除

殘差與「**槽位週轉**」一致（進場更頻繁、持倉更短、利用率反而較低——
DL-THR 的平均利用率 0.35–0.43 低於 Z-Score 的 0.45–0.57），
惟此假說需重跑回測方能直接驗證 → 列為後續研究。

:::


## 增益來源④：也不是「全資訊」這個運氣

前三項排除的是**行為**上的替代解釋。本項排除一項**問題性質**上的解釋：
DL-THR 之所以有效，會不會只是因為這個問題碰巧能把 9 個動作的報酬全部算出來？

RL-THR 保持動作選單、狀態、網路與 walk-forward 切分完全相同，只把訓練標籤
縮成「實際選中的那一個」並改採 $\varepsilon$-greedy（設計見 §3.4）。
配對底 `Grid (AGG-SSD)`，15 格等權、逐日、循環 block bootstrap（$n$=6,287）。

| 對照 | 年化Δ | 95% CI (pp) | $p$ |
| :--- | ---: | :---: | ---: |
| **DL-THR − Z-Score**（參照） | **+0.787 pp** | [+0.34, +1.27] | **0.0011** |
| RL-THR ($\varepsilon$=0.05) − Z-Score | +0.707 pp | [+0.10, +1.45] | **0.0400** |
| **RL-THR ($\varepsilon$=0.10) − Z-Score** | **+0.802 pp** | [+0.20, +1.54] | **0.0191** |
| RL-THR ($\varepsilon$=0.20→0.02) − Z-Score | +0.669 pp | [+0.08, +1.38] | **0.0450** |
| RL-THR ($\varepsilon$=0.05) − DL-THR | −0.081 pp | [−0.47, +0.34] | 0.6935 |
| **RL-THR ($\varepsilon$=0.10) − DL-THR** | **+0.015 pp** | **[−0.38, +0.45]** | **0.9409** |
| RL-THR ($\varepsilon$=0.20→0.02) − DL-THR | −0.118 pp | [−0.49, +0.28] | 0.5357 |

::: {.callout-important}

### 這一次「不顯著」確實等於「相當」

三組 $\varepsilon$ 一致：差距 −0.12 ~ +0.02 pp，$p$ = 0.54 ~ 0.94。
最有利排程下兩臂相差 **+0.015 pp**——區間 **[−0.38, +0.45]** 幾乎對稱橫跨零。

判準與命題 1 相同，結論卻相反——**看區間寬度**。此處兩端皆遠小於總增益
+0.787 pp，屬「兩端皆小」故可主張**相當**；命題 1 則是「兩端皆大」的
檢定力不足（§4.1）。

→ 增益**不來自**全資訊設定。把答案卷收走、改成真正的部分回饋，增益幾乎原封不動。
結合已證偽的逐日定位動作空間（三代真 RL，中位 Sharpe −1.1 ~ −2.3），
兩個方向指向同一結論：**關鍵是動作空間設計，不是訓練方法。**

:::

::: {.callout-warning}

### 兩個報告口徑在此給出相反的答案

改報**最佳格**（五輪獨立重跑，同一配對底）：

| 策略 | 最佳年化中位 | 全距 | 最佳 Sharpe 中位 | 全距 |
| :--- | ---: | :---: | ---: | :---: |
| DL-THR | **2.387%** | [2.341, 2.649] | **0.343** | [0.337, 0.377] |
| RL-THR $\varepsilon$=0.10 | 2.107% | [1.985, 2.192] | 0.307 | [0.288, 0.314] |
| RL-THR $\varepsilon$=0.05 | 1.897% | [1.811, 2.045] | 0.276 | [0.266, 0.294] |

DL-THR **五輪全勝且全距完全不重疊**。不矛盾：best-of-15 放大微小而系統性的優勢——
資訊量九倍的那一臂更**可靠地**產出好的極大值，即使平均水準相同；
等權組合把這個選擇效應平均掉。

本研究一律以等權組合為口徑（§3.5），故結論取「相當」。
但此處同時是一個具體警示：**同一份資料在最佳格口徑下會支持相反的結論。**

:::

**一項限制**　RL-THR 與 DL-THR 的差距同時含「資訊量僅 1/9」與「探索成本」
兩個效應，本設計無法分離——不探索就沒有樣本，這是部分回饋的定義（§3.4）。


# 4.3 組合系統：標題宣稱的那個檢定

4.1 與 4.2 各測一個成分，**都不是實務上要部署的系統**。

> **組合系統**　動態分群（252 日窗、21 日滾動、295 期）+ SSD/SDP 排序 + 共整合篩選 + **DL-THR 交易端**
>
> **傳統基準**　GICS 產業分組 + 同一排序 + 同一篩選 + **固定門檻 Z-Score**

**全期（2000–2025）**

| 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | $p$ | BH 校正 $p$ | |
| :--- | :--- | ---: | ---: | :---: | ---: | ---: | :---: |
| Agglomerative | GICS-SSD | **+1.156** | 0.455 | [+0.32, +2.01] | 0.0094 | **0.0141** | **✔** |
| HDBSCAN | GICS-SDP | **+1.025** | 0.412 | [+0.31, +1.81] | 0.0076 | **0.0141** | **✔** |
| K-means | GICS-SSD | +0.631 | 0.239 | [−0.31, +1.56] | 0.1864 | 0.1864 | ✘ |

全期三組中**兩組**於 BH 校正後達 5% 顯著。


## 組合系統（續）：2012 年後

排除 2008 金融海嘯與其後的高波動期後重跑同一組對照：

| 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | $p$ | BH 校正 $p$ | |
| :--- | :--- | ---: | ---: | :---: | ---: | ---: | :---: |
| Agglomerative | GICS-SSD | **+1.511** | 0.762 | [+0.53, +2.57] | 0.0046 | **0.0074** | **✔** |
| HDBSCAN | GICS-SDP | **+1.194** | 0.647 | [+0.36, +2.06] | 0.0049 | **0.0074** | **✔** |
| K-means | GICS-SSD | **+1.193** | 0.576 | [+0.17, +2.21] | 0.0203 | **0.0203** | **✔** |

2012 年後**三組全部顯著**，且效果量與資訊比率皆高於全期。

**合計：六組對照中五組於 BH 校正後達 5% 顯著。**
（排序準則已對齊，否則會混入分組與排序兩個變因。）

::: {.callout-important}

### 這個結論不可讀成「ML 分群有效」

**成分分解顯示分群成分單獨仍無一顯著**（下節）。組合系統之所以顯著，
是兩個方向相同的效果**相加**跨過門檻，而非分群本身被證實有效。

正確的敘述是：**完整系統顯著優於傳統基準；但把功勞歸給哪一半，資料還答不出來。**

:::


## 成分分解：改善來自哪一半？

$$(\text{分群}+\text{DL-THR}) - (\text{GICS}+\text{ZS}) = \underbrace{(\text{分群}+\text{DL-THR}) - (\text{分群}+\text{ZS})}_{\text{DL-THR 成分}} + \underbrace{(\text{分群}+\text{ZS}) - (\text{GICS}+\text{ZS})}_{\text{分群成分}}$$

**全期**

| 分群法 | 總效果 | DL-THR 成分 | ($p$) | 分群成分 | ($p$) |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Agglomerative | +1.156 | **+0.787** | **0.0011** | +0.369 | 0.363 |
| HDBSCAN | +1.025 | **+0.865** | **0.0000** | +0.160 | 0.598 |
| K-means | +0.631 | **+0.606** | **0.0060** | +0.025 | 0.954 |

**2012 年後**

| 分群法 | 總效果 | DL-THR 成分 | ($p$) | 分群成分 | ($p$) |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Agglomerative | +1.511 | **+0.743** | **0.0008** | **+0.768** | 0.122 |
| HDBSCAN | +1.194 | **+1.000** | **0.0000** | +0.195 | 0.612 |
| K-means | +1.193 | **+0.464** | **0.0200** | **+0.730** | 0.140 |

分解殘差 $\le$ 0.001 pp；兩欄即 §4.2 與 §4.1 的原表。

::: {.callout-important}

### DL-THR 成分六組全部顯著；分群成分六組全部不顯著

這是本節唯一能下的斷言。

1. **方向為正 $\ne$ 有效。** 六組分群成分的 $p$ 值介於 0.122 ~ 0.954，
   無一達 5% 顯著。2012+ 的 Agglomerative（+0.768）與 K-means（+0.730）
   在數值上已接近 DL-THR 成分，卻仍無法拒絕「分群無貢獻」。

→ 支持採用**交易端**改良（證據充分）；
   對「以分群取代產業分類」則**維持未獲支持**——不是被否證，是檢定力不足。

:::


# 4.4 命題 1 失敗原因的受控歸因

§4.1 得到的是「未獲支持」。但這個結果有兩種可能的來源，必須分開：

| | 若為真 | 可否由本研究判定 |
| :--- | :--- | :--- |
| **甲：方法本身在此環境無效** | ML 分群確實無法建立更優的搜尋空間 | 這是命題 1 想回答的 |
| **乙：本研究的實作壓抑了方法** | 方法有效，但被某個實作選擇擋住了 | **必須逐一排除，否則甲無法成立** |

::: {.callout-important}

### 三項候選的選取依據

本研究與命題 1 的動機來源 Han, He & Toh (2021) 逐項比對後，
實作差異可歸為三類，**恰好對應配對交易流程的三個階段**：

| # | 差異所在 | 具體內容 | 本研究的檢驗 |
| :--- | :--- | :--- | :--- |
| **① 形成期設計** | 選配對之前 | 特徵含 12 維 GICS one-hot、施加共整合篩選、消融矩陣缺「不分組」零點 | 2×2×2 因子設計 |
| **② 交易機制** | 選完配對之後 | 該文為群內短期反轉、$\beta$=1、發散即建倉、持有一個月；本研究為共整合均值回歸、OLS-$\beta$、$z>2$ 建倉、收斂才出場 | 四項差異**逐步**施加 |
| **③ 特徵維度** | 特徵本身 | 該文 48 動量 + 78 公司特徵；本研究連續 7 維 | 擴充至 19 維並拆解插補管道 |

**順序有其邏輯**：① 檢驗「搜尋空間怎麼切」、② 檢驗「找到後怎麼交易」、
③ 檢驗「用什麼描述一檔股票」。三者涵蓋了從原始資料到損益的完整路徑，
其餘差異（母體範圍、資料頻率）**無法以參數調整消除**，列為研究限制（§5.3）。

:::

> **三節共同的結論**：三項候選**皆使形成期結構如預期改變**
> （跨產業比例、候選池充足度、特徵維度都動了），**但績效皆未隨之改善**。
> 故乙被排除到本研究能力所及的程度——但這不等於甲成立，
> 因為 §4.1 的檢定力本身不足（§4.5）。

## 形成期實作差異：2×2×2 因子設計

與 Han et al. (2021) 的三處差異：12 維 GICS one-hot、共整合篩選、缺「不分組」零點。

| 分組 | 篩選 | 期均/20 | 跨產業% | 等權年化% |
| :--- | :---: | ---: | ---: | ---: |
| GICS 產業 | coint | 15.0 | 0.0 | −0.333 |
| AGG +one-hot | coint | 11.5 | 10.7 | −0.234 |
| AGG −one-hot | coint | 11.3 | **39.6** | −0.300 |
| 不分組 | coint | **19.4** | **75.3** | −0.245 |
| GICS 產業 | none | 20.0 | 0.0 | +0.135 |
| AGG +one-hot | none | 20.0 | 6.8 | −0.373 |
| AGG −one-hot | none | 20.0 | 25.3 | **+0.226** |
| 不分組 | none | 20.0 | 44.7 | +0.184 |

**形成期結構如預期改變，揭露兩項機制**

1. **產業先驗強度成單調階梯**：0% → 10.7% → 39.6% → 75.3%。
   實測 one-hot 使分群近鄰中的**跨產業配對距離膨脹 +77.9%**，同產業 +0.0%
2. **候選池飢餓是「分群 × 篩選」的交互作用**：分群+篩選僅 11.3–11.5/20，
   但**不分組+篩選達 19.4/20** —— 篩選僅在分群先切小候選池後才造成飢餓

**惟績效未改善**：20 項對照 BH 校正後僅 1 項顯著，且**未能在另一口徑複現**。


## 交易機制：四項差異逐步施加

| 步驟 | 該步改變 | 等權年化% | Top1 年化% |
| :--- | :--- | ---: | ---: |
| 起點 | SSD 距離 + OLS-β + $z>2$ / 126 日 | −0.373 | −0.909 |
| ② | β 改 1 等金額 | −0.177 | −0.370 |
| ③ | 選對準則改**月報酬發散** | −0.092 | +0.207 |
| ④ | 21 日窗 + 發散即建倉 + 持有至期末 | −0.094 | **+1.369** |

**方向一致但統計上不顯著**：Top1 單調改善 −0.909% → +1.369%（+2.28 pp），
惟校正前最小 $p$=0.055、BH 後無一顯著；
總效果等權 +0.28 pp ($p$=0.923)、Top1 +2.28 pp ($p$=0.705)。

::: {.callout-important}

**完整復刻 Han et al. 的交易端後，等權年化仍為 −0.094%**
—— 與原文報告的 **24.8%** 相差**兩個數量級**。

:::


## 特徵維度與插補管道

**特徵維度**：自 SEC companyfacts 解出 40 個 Green et al. 特徵，
**12 個**覆蓋率 >70%，連續維度 7 → 19。2012+ 逐步鏈：

| 步驟 | 年化Δ(pp) | BH 校正 $p$ |
| :--- | ---: | ---: |
| one-hot → 0 | −0.171 | 0.602 |
| 產業插補 → 全域插補 | +0.481 | 0.602 |
| **7 維 → 19 維** | −0.482 | 0.602 |

兩大效果幾乎抵消（淨 −0.001 pp）。對 GICS 為 +0.283 pp，**未優於** 7 維版的 +0.455 pp。

⚠️ 這兩個中間步驟的**符號不穩定**：補齊原始價快取前為 −0.716／+0.685，
補齊後皆反轉，而淨效果與「全部不顯著」不變——不顯著的點估計不宜作方向性解讀。
→ **這些特徵替代了產業先驗，但未超越它。**

::: {.callout-important}

### 產業中位數插補是第三條產業資訊管道

結構性特徵 2012+ 缺失 30–50%，插補值等同為缺資料股票加上產業標籤。
**固定特徵、僅改插補方式**：

| 臂 | 產業插補 | 全域插補 | 差 |
| :--- | ---: | ---: | ---: |
| AGG-BASE | 9.5% | 16.5% | +7.0 |
| **AGG-STRUCT** | **15.7%** | **59.3%** | **+42.8** |

影響在 STRUCT 臂（缺失所在）遠大於 BASE 臂。

**已封存的 F09 消融因此是在「處理幾乎未施加」下執行**；
以全域插補重驗後其「無貢獻」結論**仍成立**（BH 後 0/6 顯著），
並由**弱 null 升級為強 null**。

:::


# 4.5 核心發現：形成期的改良難以驗證，且難在設計

本章對兩個層次施加了**形式相同**的受控對照，卻只在其中一層得到可歸因的答案。
這並非偶然，而是兩層比較的設計差異所致。

## 三項可陳述的結論

**其一，交易期層的顯著並非檢定力僥倖。**
點估計中位 +0.787 pp 明顯大於其自身 MDE 0.60 pp；五組信賴區間完全落在零的右側，
最保守下界仍為 +0.20 pp。

**其二，形成期層的不顯著幾乎不帶資訊。**
若 ML 分群的真實優勢為 0.3 pp（已足以翻轉參照臂的損益方向），
本檢定偵測到它的機率僅 **11%**。故 §4.1 的九組 null 對「命題 1 是真是假」
幾乎不具鑑別力。

**其三，不對等來自設計，而非資料量。**
$SE$ 相差 **1.89 倍**，等效於形成期層需 **3.6 倍**樣本期間。
若要求把 MDE 壓到 0.3 pp（能看見經濟上實質的差異），形成期層需約 **14 倍**樣本、
**逾三個世紀**的日資料；交易期層亦需約 4 倍。

::: {.callout-important}

### 這是結構性限制

更換分群演算法、增加特徵維度、改良插補方式**皆不影響它**。

唯一的出路是在形成期層構造**配對設計**——使兩臂持有同一批標的而僅變動分組。
本研究未能設計出這樣的對照，列為後續研究的首要方法論課題。

:::


## 輔證：跨產業比例大幅變動，未伴隨績效變動

綜合 §4.1 與 §4.4 的全部消融，本研究在六種設定下改變了「交易哪些配對」：

| 設定 | 跨產業配對比例 | 2012+ 年化 |
| :--- | ---: | ---: |
| GICS 產業分組 | **0.0%** | −1.43% |
| Agglomerative + one-hot | 8.6% | −0.97% |
| Agglomerative − one-hot | 31.4% | −1.14% |
| Agglomerative + 全域插補 | 54.5% | −0.66% |
| Agglomerative + 12 公司特徵 | 62.8% | −1.14% |
| **完全不分組** | **69.4%** | −1.84% |

涵蓋 **one-hot 權重、插補方式、特徵集、分組方法**四個維度，
跨產業比例自 **0% 變動至 69%**，而 2012+ 績效全部落在 **−1.84% ~ −0.66%**。

::: {.callout-warning}

### 但這項證據的強度必須據實界定

六種設定的績效**全距為 1.18 pp**，而 §4.1 中**單一組對照**的區間寬度即為
**1.18 ~ 1.91 pp**——**全部六種設定的離散度，尚不及一組對照的區間寬度**。

這與「配對選擇不影響報酬」一致，但同樣與「本實驗看不見 1 pp 以下的效果」一致。
**兩者無法由本表區分。**

:::

> **可以陳述的是**：在跨產業配對比例 0% 至 69% 的範圍內，
> 本研究未觀察到本實驗解析度所能偵測的績效差異。
>
> **不可陳述的是**：「選哪些配對不決定報酬」或「形成期搜尋空間的邊際貢獻趨近於零」。
> 本研究並未施加等價檢定，依 §3.5.3 自訂的標準，此類主張須有實質等價邊界的
> 證據支持，本表不提供該證據。

因此命題 1 的未獲支持，既不應理解為「某種分群演算法不夠好」，
亦不應理解為「形成期不重要」，而應理解為：
**在本研究的設計與樣本下，形成期層的改良無法被驗證。**


# 4.6 風險評估

## Regime 分層穩健性

口徑同全章：**15 格等權組合**，且**逐格對齊兩臂**（只取兩交易端都有的參數格）
→ Z-Score 與 DL-THR 之間的唯一變因仍是交易端。

| 配對底 | 交易端 | Calm | Normal | Turbulent | Bull | Bear |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: |
| Agglomerative | Z-Score | −0.81 | −0.60 | 0.79 | −0.23 | 0.41 |
| Agglomerative | **DL-THR** | **−0.38** | **−0.36** | **0.95** | **+0.08** | **0.59** |
| HDBSCAN | Z-Score | −0.64 | −0.55 | 0.63 | −0.17 | 0.29 |
| HDBSCAN | **DL-THR** | **−0.20** | **−0.09** | **0.66** | **+0.15** | **0.46** |
| K-means | Z-Score | −0.71 | −0.60 | 0.41 | −0.22 | −0.02 |
| K-means | **DL-THR** | **−0.55** | **−0.12** | **0.48** | **−0.03** | **0.29** |
| GICS-SSD（傳統） | Z-Score | −1.24 | −0.52 | 0.54 | −0.44 | 0.31 |
| GICS-SSD（傳統） | **DL-THR** | **−0.72** | **−0.15** | **0.70** | **−0.04** | **0.52** |
| GICS-SDP（傳統） | Z-Score | −0.69 | −0.77 | 0.68 | −0.27 | 0.31 |
| GICS-SDP（傳統） | **DL-THR** | **−0.49** | **−0.32** | **0.75** | **−0.01** | **0.47** |

**其一：DL-THR 的增益具 regime 穩健性** —— 25 格中改善 **25 格**、劣化 **0 格**

**其二：獲利高度集中於動盪期** —— 十組配置在 Calm 期**全部**為負 Sharpe（−1.24 ~ −0.20）。
DL-THR 可減輕平靜期虧損但無法反轉。

> 策略實質上在**賣出波動率的均值回歸選擇權**，低波動環境即為其不利環境。


## 交易成本敏感度：餘裕薄到接近於零

成本模型可解析求解：進出場費用 = friction × 名目額，且名目額恰等於每配對資金，
故單邊 break-even $c^* = $ 現行單邊費 + 淨利 / $\Sigma$名目額。
口徑同上頁：15 格等權、逐格對齊兩臂。

| 配對底 | Z-Score 往返 BE% | DL-THR 往返 BE% | 差（bps） | Z 餘裕（bps） | DL 餘裕（bps） |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Agglomerative | 0.582 | **0.601** | +1.9 | +0.2 | **+2.1** |
| HDBSCAN | 0.582 | **0.604** | +2.1 | +0.2 | **+2.4** |
| K-means | 0.562 | **0.591** | +2.9 | **−1.8** | +1.1 |
| GICS-SSD（傳統） | 0.568 | **0.594** | +2.7 | **−1.2** | +1.4 |
| GICS-SDP（傳統） | 0.576 | **0.592** | +1.6 | **−0.4** | +1.2 |

**其一：DL-THR 在五種配對底上一致提高 break-even**（+1.6 ~ +2.9 bps）
——與上頁的 regime 結果同向，是交易端改良最實際的一項佐證。

::: {.callout-important}

### 其二：成本餘裕薄到接近於零

十組配置的餘裕介於 **−1.8 ~ +2.4 bps**，且 **K-means、GICS-SSD、GICS-SDP
三個 Z-Score 臂已經為負**——在現行假設（往返 0.58%）下它們本來就不獲利。

DL-THR 把五個配對底全部推回正值，但幅度僅 1~2 bps，
**遠小於成本假設本身的不確定性**。

:::


## 絕對績效：等權口徑下無一顯著

$H_0$：平均日報酬 = 0，**無對照組**。口徑同全章：15 格等權組合
→ **沒有東西可挑，故無選擇偏誤需要校正**。

| 配對底 | 交易端 | 年化 | 95% CI (pp) | $p$ |
| :--- | :--- | ---: | :---: | ---: |
| Agglomerative | Z-Score | −0.03% | [−1.07, +1.24] | 0.965 |
| Agglomerative | DL-THR | +0.68% | [−0.48, +2.12] | 0.303 |
| HDBSCAN | Z-Score | −0.05% | [−1.25, +1.48] | 0.945 |
| HDBSCAN | DL-THR | +0.77% | [−0.50, +2.41] | 0.299 |
| K-means | Z-Score | −0.36% | [−1.35, +0.77] | 0.516 |
| K-means | DL-THR | +0.27% | [−0.83, +1.53] | 0.659 |

**六個區間全部橫跨零。** 三個 Z-Score 臂點估計皆負、三個 DL-THR 臂皆正，
方向與命題 2 一致。

::: {.callout-warning}

### 本表與 §4.2 的數字**不可直接相減**

本表的日報酬採**複利**口徑（`Daily_Delta` / 前一日權益），與引擎依當期權益
配置部位的實際行為一致；§4.2 的差分檢定則採**單利**口徑（日損益金額 / 期初資金）。

兩個複利序列的差，不等於同一組資料在單利口徑下的差分——
以 Agglomerative 為例，本表相減得 **+0.71 pp**，§4.2 為 **+0.79 pp**。
方向與量級一致，差距源於口徑而非資料，**故不作為自洽性檢核使用**。

:::

::: {.callout-important}

檢定力遠低於相對比較：DL-THR 臂區間寬達 **2.4 ~ 2.9 pp**，
資料無法區分「年化 +2%」與「年化 −1%」。

**本研究不主張任何單一策略的絕對獲利能力具統計顯著性。**

:::

> **附錄的 Deflated Sharpe**：若改報「網格最佳格」，數字較高但內含 15 選 1 的偏誤。
> 以實地清點的試驗宇宙 $N$ = 110 計，共同門檻 $SR_0$ = 0.408（年化），
> 而上表六族中最高的 $SR$ 僅 **0.392**——純靠運氣就該達到的水準沒有一組達到，
> DSR 全數落在 **0.256 ~ 0.467**，無一通過 0.95。
>
> $N$ = 110 由程式實地清點相異策略數，含附錄 A 為隔離前行研究差異而新增的三條
> `HSU25 (…-REV)`——結論為負的試驗仍佔用一次試驗，**試驗數不因結果不利而豁免**。
> 涵蓋全資料庫的 **98 個策略族亦無一通過 0.95**，最高者 $SR$ = 0.471、$DSR$ = 0.632。
>
> **此表不構成正文的主張。**


## 相對顯著、絕對不顯著為何不矛盾

| | 逐日差分 bootstrap | 絕對 bootstrap |
| :--- | :--- | :--- |
| $H_0$ | 兩交易端績效相同 | 策略平均日報酬為零 |
| 對照物 | 同配對、同參數格的 Z-Score | **零** |
| 主張性質 | **相對** | **絕對** |

配對設計消去共同市場風險 → 訊噪比大幅提高；
絕對檢定無此對照 → 必須從市場噪音中直接辨識訊號。

**檢定力受限於低 Sharpe，而非樣本長度**（$t \approx SR\sqrt{\text{年數}}$，
樣本 6,287 個交易日 ≈ 24.9 年）：

| 配對底（DL-THR 臂） | 年化 Sharpe | $SR\sqrt{24.9}$ | 實測 $t$ | 達 $p<0.05$ 所需年數 |
| :--- | ---: | ---: | ---: | ---: |
| HDBSCAN | 0.209 | 1.04 | 1.04 | **88** |
| Agglomerative | 0.206 | 1.03 | 1.03 | 91 |
| K-means | 0.088 | 0.44 | 0.44 | 493 |

理論值與實測值一致 → 檢定力確實由 Sharpe 決定。
以最佳的 HDBSCAN 臂計，欲使絕對績效達顯著約需 **88 年**樣本。
此為量級判斷而非精確預測：Sharpe 本身即為估計值，其抽樣誤差會連帶放大所需年數。

> **本研究的檢定定位**：命題 2 為**方法之相對優劣**的假設檢定，證據充分；
> 絕對獲利能力在本研究的樣本長度下**原則上就無法檢定**——
> 這並非樣本不足，而是該量級的 Sharpe 與可得資料長度之間的**結構性落差**。


## 本章小結

::: {.callout-important}

**組合系統顯著優於傳統基準。** 六組對照中五組於 BH-FDR 校正後達 5% 顯著
（+0.63 ~ +1.51 pp）。惟此為**相對**主張——同一系統的絕對績效在等權口徑下
**全部不顯著**，兩者虛無假設不同（§4.6.4）。

**改善可歸因於交易端，不可歸因於分群層。** 成分分解六組中，
DL-THR 成分全部顯著、分群成分全部不顯著（$p$ = 0.122 ~ 0.954）。
惟**方向為正不等於有效**。

**命題 2 獲得支持**，五種配對底全部顯著且信賴區間完全落在零的右側，
三項替代解釋（拉高門檻、篩選配對、減少曝險）皆已排除；
惟門檻管道在 HDBSCAN 底下複製 28.4% 且達顯著，已據實揭露。

**命題 1 未獲支持**，且失敗原因不可歸因於本研究的實作選擇——
形成期設計、交易機制、特徵維度三項候選皆已受控檢驗且皆非原因。

:::

::: {.callout-note}

### 核心發現：形成期層的改良難以驗證，難在**設計**而非資料量

兩臂持不同配對使 $SE$ 較交易期層大 **1.89 倍**（等效需 **3.6 倍**樣本）；
對 0.3 pp 的真實效果，本檢定的檢定力僅 **11%**。

跨產業配對比例自 0% 至 69% 的六種設定未觀察到可偵測的績效差異，
惟其全距（1.18 pp）尚不及單一對照的區間寬度 → **此為輔證，非等價證據**。

:::

**風險面**：獲利集中於動盪期（十組配置在 Calm 期全部為負 Sharpe）、
DL-THR 在五種配對底上一致提高 break-even（+1.6~+2.9 bps）但**餘裕本身僅 −1.8~+2.4 bps**、
且等權組合的絕對績效六組全部不顯著（區間寬 2–3.5 pp）。

> 本研究的結論限於**方法間的相對比較**，
> **不宣稱策略具可實現之超額報酬**。
